<table>
    <tr>
      <td>Minería de datos y paradigma BigData - Facultad de Informática - UCM
      </td>
      <td>
      <img src="https://biblioteca.ucm.es/data/cont/media/www/pag-88746//escudo.jpg" width=50/>
      </td>
     </tr>
</table>

# Práctica: Análisis de tráfico IoT con PySpark
### Pablo C. Cañizares

En esta práctica aprenderás a trabajar con **PySpark** para analizar un dataset real de tráfico de red en un entorno IoT. El notebook cubre tres bloques:

1. **Parte I — DataFrames** (10 ejercicios): carga, exploración, filtrado, agregación y transformaciones.
2. **Parte II — Machine Learning con MLlib**: clasificación automática de ataques con Random Forest.
3. **Parte III — Introducción a MLflow**: registro de experimentos, métricas y modelos.

> **Requisitos:** Este notebook está diseñado para ejecutarse en **Databricks Free Edition**. La variable `spark` ya está disponible automáticamente.

---

## Dataset: RT_IOT2022

El dataset **RT_IOT2022** contiene flujos de tráfico de red capturados en un entorno IoT. Cada fila es un flujo de red con **83 características** y una **etiqueta** que indica si es tráfico normal o un tipo de ataque.

Las columnas que más usaremos son:

| Columna | ¿Qué es? | Ejemplo |
|---|---|---|
| `proto` | Protocolo de red (tcp, udp, icmp) | tcp |
| `service` | Servicio detectado (mqtt, http, dns…) | mqtt |
| `flow_duration` | Duración del flujo en segundos | 32.01 |
| `fwd_pkts_tot` | Paquetes enviados (origen → destino) | 9 |
| `bwd_pkts_tot` | Paquetes recibidos (destino → origen) | 5 |
| `flow_pkts_per_sec` | Paquetes por segundo del flujo | 0.437 |
| `pkt_len_avg` | Longitud media de paquete (bytes) | 126.5 |
| `flow_bytes_s` | Bytes por segundo del flujo | 1725.65 |
| `Attack_type` | Etiqueta: tipo de ataque o "normal" | MQTT_Publish |

> **Carga en Databricks:** Sube `RT_IOT2022.csv` a tu volumen recien creado (si no lo has creado aún: Catalog > New volume).

---

## Setup

In [0]:
# Importaciones necesarias
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# Comprobamos que Spark está disponible
print(f"PySpark versión: {spark.version}")

# Ruta al CSV — ajusta si lo has subido a otra ubicación
CSV_PATH = "/Volumes/workspace/default/test/RT_IOT2022.csv"


PySpark versión: 4.1.0


---

# Parte I: Análisis con DataFrames (10 ejercicios)

Un **DataFrame** en Spark es como una tabla de base de datos distribuida: tiene columnas con nombre y tipo, y Spark optimiza automáticamente las operaciones gracias a su motor interno llamado **Catalyst**.

Vamos a aprender las operaciones fundamentales: cargar datos, filtrar, agrupar, transformar y combinar.

---

### Ejercicio 1: Cargar el CSV y explorarlo

Carga el fichero CSV como DataFrame y responde:
- ¿Cuántas filas tiene el dataset?
- ¿Cuántas columnas?
- ¿Qué tipos de datos ha inferido Spark?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

# Cargamos el CSV con cabecera y detección automática de tipos
df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CSV_PATH)
)

# Vemos el número de filas y columnas
print(f"Filas: {df.count()}")
print(f"Columnas: {len(df.columns)}")

# Muestra el nombre y tipo de cada columna
df.printSchema()


Filas: 123117
Columnas: 85
root
 |-- no: integer (nullable = true)
 |-- id.orig_p: integer (nullable = true)
 |-- id.resp_p: integer (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- fwd_pkts_tot: integer (nullable = true)
 |-- bwd_pkts_tot: integer (nullable = true)
 |-- fwd_data_pkts_tot: integer (nullable = true)
 |-- bwd_data_pkts_tot: integer (nullable = true)
 |-- fwd_pkts_per_sec: double (nullable = true)
 |-- bwd_pkts_per_sec: double (nullable = true)
 |-- flow_pkts_per_sec: double (nullable = true)
 |-- down_up_ratio: double (nullable = true)
 |-- fwd_header_size_tot: integer (nullable = true)
 |-- fwd_header_size_min: integer (nullable = true)
 |-- fwd_header_size_max: integer (nullable = true)
 |-- bwd_header_size_tot: integer (nullable = true)
 |-- bwd_header_size_min: integer (nullable = true)
 |-- bwd_header_size_max: integer (nullable = true)
 |-- flow_FIN_flag_count: integer (n

---

### Ejercicio 2: Limpiar nombres de columnas

Algunas columnas tienen **puntos** en el nombre (`id.orig_p`, `active.min`...) que causan problemas al acceder con `col()`. Reemplázalos por guiones bajos.

> **Concepto clave:** `withColumnRenamed(antiguo, nuevo)` cambia el nombre de una columna.

In [0]:
# Itera sobre las columnas y renombralas, reemplazando '.' por '_'
for col_name in df.columns:
    new_name = col_name.replace('.', '_')
    df = df.withColumnRenamed(col_name, new_name)

# Comprobacion: mostramos las 10 primeras columnas
print("Primeras 10 columnas:", df.columns[:10])


Primeras 10 columnas: ['no', 'id_orig_p', 'id_resp_p', 'proto', 'service', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot']


---

### Ejercicio 3: Estadísticas descriptivas

Usa `.summary()` para obtener las estadísticas básicas (media, desviación, mínimo, máximo…) de las columnas numéricas más importantes.

> **Concepto clave:** `.select()` elige columnas, `.summary()` calcula estadísticas.

In [0]:
# Seleccionamos columnas numéricas clave y mostramos sus estadísticas
display(
    df.select("flow_duration", "fwd_pkts_tot", "bwd_pkts_tot",
              "flow_pkts_per_sec", "payload_bytes_per_second")
      .summary()
)


summary,flow_duration,fwd_pkts_tot,bwd_pkts_tot,flow_pkts_payload_avg,payload_bytes_per_second
count,123117,123117,123117,123117,123117
mean,3.8095657699983643,2.2688255886676902,1.9095088411835897,65.01019417995076,4.1053451791945435E7
stddev,130.0054075187572,22.336564997056605,33.01831059054765,50.412145014640664,4.4857059625514686E7
min,0.0,0,0,0.0,0.0
25%,1.0E-6,1,1,60.0,2575.0246899999997
50%,4.0E-6,1,1,60.0,2.9606851764706E7
75%,5.0E-6,1,1,60.0,5.592405333333299E7
max,21728.335578,4345,10112,1156.084685,1.2582912E8


Una vez modificados los nombres de las variables y limpiado algunos de los parámetros, vamos a analizar las variables numéricas, ver cuales son las categóricas. Finalmente veremos una matriz de correlacion

In [0]:
from pyspark.sql.types import NumericType
from pyspark.sql import Row

# Separar numéricas y categóricas
num_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
cat_cols = [c for c in df.columns if c not in num_cols]

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

Numéricas: ['no', 'id_orig_p', 'id_resp_p', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_FIN_flag_count', 'flow_SYN_flag_count', 'flow_RST_flag_count', 'fwd_PSH_flag_count', 'bwd_PSH_flag_count', 'flow_ACK_flag_count', 'fwd_URG_flag_count', 'bwd_URG_flag_count', 'flow_CWR_flag_count', 'flow_ECE_flag_count', 'fwd_pkts_payload_min', 'fwd_pkts_payload_max', 'fwd_pkts_payload_tot', 'fwd_pkts_payload_avg', 'fwd_pkts_payload_std', 'bwd_pkts_payload_min', 'bwd_pkts_payload_max', 'bwd_pkts_payload_tot', 'bwd_pkts_payload_avg', 'bwd_pkts_payload_std', 'flow_pkts_payload_min', 'flow_pkts_payload_max', 'flow_pkts_payload_tot', 'flow_pkts_payload_avg', 'flow_pkts_payload_std', 'fwd_iat_min', 'fwd_iat_max', 'fwd_iat_tot', 'fwd_iat_avg'

---

### Ejercicio 4: Filtrar registros

a) ¿Cuántos registros usan el protocolo **TCP**?

b) ¿Cuántos ataques de tipo **MQTT_Publish** hay?

> **Concepto clave:** `.filter(condición)` devuelve solo las filas que cumplen la condición. `.count()` cuenta cuántas filas hay.

In [0]:
# a) Filtrar por protocolo TCP
tcp_df = df.filter(col("proto") == "tcp")
print(f"Registros TCP: {tcp_df.count()}")

# b) Filtrar por tipo de ataque MQTT_Publish
mqtt_df = df.filter(col("Attack_type") == "MQTT_Publish")
print(f"Ataques MQTT_Publish: {mqtt_df.count()}")


Registros TCP: 110427
Ataques MQTT_Publish: 4146


---

### Ejercicio 5: Valores únicos y ordenación

a) Obtén la lista de **tipos de ataque** distintos que hay en el dataset, ordenados alfabéticamente.

b) Muestra los **5 flujos con mayor duración**. Para visualizar de una forma adecuada el resultado, selecciona las features: `proto`, `service`, `flow_duration`, `fwd_pkts_tot`, `Attack_type`.

> **Conceptos clave:** `.distinct()` elimina duplicados, `.orderBy()` ordena, `.limit(n)` toma los primeros n registros.

In [0]:
# a) Tipos de ataque distintos
tipos_ataque_df = df.select("Attack_type").distinct().orderBy("Attack_type")

print(f"Hay {tipos_ataque_df.count()} tipos de ataque distintos")
display(tipos_ataque_df)


Hay 12 tipos de ataque distintos


Attack_type
ARP_poisioning
DDOS_Slowloris
DOS_SYN_Hping
MQTT_Publish
Metasploit_Brute_Force_SSH
NMAP_FIN_SCAN
NMAP_OS_DETECTION
NMAP_TCP_scan
NMAP_UDP_SCAN
NMAP_XMAS_TREE_SCAN


In [0]:
# b) Top 5 flujos con mayor duración
top5 = (df.select("proto", "service", "flow_duration", "fwd_pkts_tot", "Attack_type")
          .orderBy(col("flow_duration").desc())
          .limit(5))

display(top5)


proto,service,flow_duration,fwd_pkts_tot,Attack_type
udp,-,21728.335578,4345,Wipro_bulb
tcp,ssl,18761.401291,704,Wipro_bulb
udp,-,17747.121108000007,3549,Wipro_bulb
tcp,ssl,17732.696969999994,671,Wipro_bulb
udp,-,9433.886888,1887,Wipro_bulb


---

### Ejercicio 6: Agrupar y contar

Cuenta cuántos registros hay **por cada tipo de ataque**, ordenados de más a menos frecuente.

> **Concepto clave:** `.groupBy("columna").count()` agrupa y cuenta. Es la operación más habitual en análisis de datos.

In [0]:
# Registros por tipo de ataque
ataques_conteo = (df.groupBy("Attack_type")
                    .count()
                    .orderBy(col("count").desc()))

display(ataques_conteo)


Attack_type,count
DOS_SYN_Hping,94659
Thing_Speak,8108
ARP_poisioning,7750
MQTT_Publish,4146
NMAP_UDP_SCAN,2590
NMAP_XMAS_TREE_SCAN,2010
NMAP_OS_DETECTION,2000
NMAP_TCP_scan,1002
DDOS_Slowloris,534
Wipro_bulb,253


---

### Ejercicio 7: Agregaciones múltiples

Agrupa por **protocolo** (`proto`) y calcula para cada uno:
- Número de registros
- Duración media del flujo
- Máximo de paquetes forward

> **Concepto clave:** `.agg()` permite calcular varias funciones a la vez: `F.count()`, `F.mean()`, `F.max()`, `F.min()`, `F.sum()`…

In [0]:
# Estadísticas por protocolo
stats_proto = (df.groupBy("proto")
                 .agg(
                     F.count("*").alias("num_registros"),
                     F.mean("flow_duration").alias("duracion_media"),
                     F.max("fwd_pkts_tot").alias("max_fwd_pkts")
                 ))

display(stats_proto)


proto,num_registros,duracion_media,max_fwd_pkts
tcp,110427,3.48,2166
udp,12633,6.5,4345
icmp,57,55.39,903


---

### Ejercicio 8: Crear columnas nuevas — Nivel de riesgo

Crea una nueva columna llamada `riesgo` que clasifique cada flujo según su tasa de bytes por segundo (`payload_bytes_per_second`):

| Condición | Riesgo |
|---|---|
| `payload_bytes_per_second` > 10 000 | ALTO |
| `payload_bytes_per_second` entre 1 000 y 10 000 | MEDIO |
| `payload_bytes_per_second` ≤ 1 000 | BAJO |

Después, cuenta cuántos flujos hay de cada nivel de riesgo.

> **Concepto clave:** Combina `woithColumn` con `F.when(cond, valor).otherwise(valor)`, que funciona como un `if-elif-else` para columnas.

In [0]:
# Crear la columna de riesgo
df_riesgo = df.withColumn(
    "riesgo",
    F.when(col("payload_bytes_per_second") > 10000, "ALTO")
     .when(col("payload_bytes_per_second") > 1000, "MEDIO")
     .otherwise("BAJO")
)

# Contar por nivel de riesgo
display(df_riesgo.groupBy("riesgo").count().orderBy("riesgo"))


riesgo,count
ALTO,86706
BAJO,28940
MEDIO,7471


---

### Ejercicio 9: Combinar tablas con join

Imagina que un analista de seguridad nos da una tabla con la **severidad** de cada tipo de ataque. Queremos añadir esa información a nuestro dataset mediante un **join**.

> **Concepto clave:** `df1.join(df2, on="columna_comun", how="left")` combina dos DataFrames por una columna compartida. Usamos `"left"` para mantener todos los registros del dataset original aunque no tengan severidad asignada.

In [0]:
# Tabla de severidad por tipo de ataque
severidad_data = [
    ("MQTT_Publish", "ALTA"),
    ("Thing_Speak", "MEDIA"),
    ("Wipro_bulb", "MEDIA"),
    ("ARP_poisioning", "CRITICA"),
    ("DDOS_Slowloris", "CRITICA"),
    ("DOS_SYN_Hping", "CRITICA"),
    ("Metasploit_Brute_Force", "CRITICA"),
    ("Metasploit_Brute_Force_SSH", "CRITICA"),
    ("NMAP_TCP_scan", "ALTA"),
    ("NMAP_FIN_SCAN", "ALTA"),
    ("NMAP_HTTP_SCAN", "ALTA"),
    ("NMAP_OS_DETECTION", "ALTA"),
    ("NMAP_UDP_SCAN", "ALTA"),
    ("NMAP_XMAS_TREE_SCAN", "ALTA"),
]

# Crea el dataframe de severidad, con columnas Attack_type y severidad.
df_severidad = spark.createDataFrame(severidad_data, ["Attack_type", "severidad"])

# Join: añadimos la severidad al dataset principal
df_enriquecido = df.join(df_severidad, on="Attack_type", how="left")

# Comprobamos el resultado
display(
    df_enriquecido
    .groupBy("severidad")
    .count()
    .orderBy("severidad")
)


severidad,count
ALTA,11776
CRITICA,102980
MEDIA,8361


---

### Ejercicio 10: Consultas SQL

Spark permite escribir consultas SQL directamente sobre DataFrames. Solo necesitas registrar el DataFrame como una **vista temporal** y usar `spark.sql()`.

Obtén los 5 tipos de ataque con mayor **duración media** de flujo. 

Para ello, construye una consulta SQL que:
- trabaje sobre la vista trafico_iot,
- agrupe por Attack_type,
- calcule el total de registros por grupo,
- calcule la media de flow_duration,
- calcule la media de payload_bytes_per_second,
- redondee ambas medias a 2 decimales,
- ordene por la duración media de mayor a menor,
- y muestre solo los 5 primeros resultados.”

> **Concepto clave:** `createOrReplaceTempView("nombre")` crea una tabla temporal. Luego puedes usar SQL estándar.

In [0]:
# Registrar como tabla temporal
df.createOrReplaceTempView("trafico_iot")

# Consulta SQL
resultado_sql = spark.sql("""
    SELECT 
        Attack_type,
        COUNT(*) AS total_registros,
        ROUND(AVG(flow_duration), 2) AS duracion_media,
        ROUND(AVG(payload_bytes_per_second), 2) AS bytes_por_segundo_media
    FROM trafico_iot
    GROUP BY Attack_type
    ORDER BY duracion_media DESC
    LIMIT 5
""")

display(resultado_sql)


Attack_type,total,duracion_media,bytes_por_seg_medio
Wipro_bulb,253,586.85,22386.29
MQTT_Publish,4146,43.4,65.52
ARP_poisioning,7750,15.89,222955.02
DDOS_Slowloris,534,14.7,93635.9
Metasploit_Brute_Force_SSH,37,3.01,13894.89


---

# Parte II: Clasificación de ataques con MLlib

Ahora vamos a entrenar un modelo de **Machine Learning** para que clasifique automáticamente el tipo de ataque a partir de las características del flujo de red.

**¿Qué modelo usamos?** Un **Random Forest** (bosque aleatorio): un conjunto de muchos árboles de decisión que "votan" la clase final. Funciona muy bien con datos tabulares y es fácil de interpretar.

**¿Qué pasos seguiremos?**
1. Seleccionar las columnas que usará el modelo (features)
2. Transformar textos a números (los modelos solo entienden números)
3. Dividir en datos de entrenamiento y de test
4. Entrenar y evaluar

---

## Paso 1: Selección de features

In [0]:
from pyspark.sql import functions as F

# Columnas numéricas candidatas
feature_cols_numeric = [
    "flow_duration", "fwd_pkts_tot", "bwd_pkts_tot",
    "fwd_data_pkts_tot", "bwd_data_pkts_tot",
    "fwd_pkts_per_sec", "bwd_pkts_per_sec", "flow_pkts_per_sec",
    "down_up_ratio", "fwd_header_size_tot", "bwd_header_size_tot",
    "flow_FIN_flag_count", "flow_SYN_flag_count", "flow_RST_flag_count",
    "payload_bytes_per_second"
]

# Columnas categóricas
feature_cols_categorical = ["proto", "service"]

# Objetivo
target_col = "Attack_type"

# Columnas solicitadas
requested_cols = feature_cols_numeric + feature_cols_categorical + [target_col]

# Selección segura
df_ml = df.select(*requested_cols)

# Ver nulos antes de eliminarlos
display(
    df_ml.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_ml.columns
    ])
)

# Si luego quieres limpiar:
df_ml = df_ml.dropna()

print(f"Registros disponibles para ML: {df_ml.count()}")

flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,bwd_pkts_per_sec,flow_pkts_per_sec,down_up_ratio,fwd_header_size_tot,bwd_header_size_tot,flow_FIN_flag_count,flow_SYN_flag_count,flow_RST_flag_count,payload_bytes_per_second,proto,service,Attack_type
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Registros disponibles para ML: 123117


---

## Paso 2: Construir el Pipeline

Un **Pipeline** encadena varios pasos de transformación y el modelo en una sola secuencia. Esto tiene dos ventajas: evita errores y facilita reproducir el proceso.

Nuestro pipeline:
1. `StringIndexer` → convierte texto a números (proto, service, Attack_type). Para ello, utiliza un StringIndexer.
2. `VectorAssembler` → junta todas las features en un único vector
3. `RandomForestClassifier` → el modelo que aprende a clasificar

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

# Convertir columnas de texto a números
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in feature_cols_categorical
]

# Convertir la etiqueta (Attack_type) a número
label_indexer = StringIndexer(inputCol="Attack_type", outputCol="label", handleInvalid="keep")

# Unir todas las features en un vector
todas_las_features = feature_cols_numeric + [c + "_idx" for c in feature_cols_categorical]
assembler = VectorAssembler(inputCols=todas_las_features, outputCol="features", handleInvalid="skip")

# Modelo Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=8,
    seed=42
)

# Encadenamos todo en un Pipeline
pipeline = Pipeline(stages=indexers + [label_indexer, assembler, rf])

print(f"Pipeline con {len(pipeline.getStages())} etapas:")
for i, etapa in enumerate(pipeline.getStages()):
    print(f"  {i+1}. {type(etapa).__name__}")


Pipeline con 5 etapas:
  1. StringIndexer
  2. StringIndexer
  3. StringIndexer
  4. VectorAssembler
  5. RandomForestClassifier


---

## Paso 3: Dividir datos y entrenar

In [0]:
# 80% para entrenar, 20% para evaluar
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Datos de entrenamiento: {train_df.count()}")
print(f"Datos de test: {test_df.count()}")


Datos de entrenamiento: 98780
Datos de test: 24337


---

## Paso 4: Predicciones y evaluación

In [0]:

# Vamos a utilizar solo 10% de los datos para evitar cache overflow
train_df_small = train_df.sample(fraction=0.1, seed=42)
print(f"Entrenando con {train_df_small.count()} registros (10% del total)")

# Usar un modelo más pequeño
rf_small = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=10,
    maxDepth=3,
    seed=42
)

pipeline_small = Pipeline(stages=indexers + [label_indexer, assembler, rf_small])
modelo = pipeline_small.fit(train_df_small)
predicciones = modelo.transform(test_df)

# Veamos algunas predicciones
display(
    predicciones
    .select(target_col, "label", "prediction", "probability")
    .limit(10)
)


Entrenando con 9907 registros (10% del total)
¡Modelo entrenado!


Attack_type,label,prediction,probability
MQTT_Publish,3.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5184377497531225"",""0.0064224118018203534"",""0.005527004486641804"",""0.0"",""0.10467381293436387"",""0.16010592785261107"",""0.15921823873964897"",""0.03950561001003762"",""0.003488983092347092"",""3.3516704204461495E-4"",""0.0"",""0.0022850942873621394"",""0.0""]}"
MQTT_Publish,3.0,5.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3253290779891408"",""0.006409927407313487"",""0.013112123235492134"",""0.0"",""0.1106520738039291"",""0.33370676041505315"",""0.15544378054511104"",""0.037105096826632086"",""0.011354015931732644"",""3.3516704204461495E-4"",""0.0"",""0.006551976803550854"",""0.0""]}"
MQTT_Publish,3.0,5.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3253290779891408"",""0.006409927407313487"",""0.013112123235492134"",""0.0"",""0.1106520738039291"",""0.33370676041505315"",""0.15544378054511104"",""0.037105096826632086"",""0.011354015931732644"",""3.3516704204461495E-4"",""0.0"",""0.006551976803550854"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"
Thing_Speak,1.0,4.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.07002126151665485"",""0.03256665349786945"",""0.2754927929689891"",""0.0"",""0.40287665489235985"",""0.10536256076084856"",""0.10341725686857124"",""0.007711262349811923"",""0.0"",""8.816151294763087E-4"",""0.0"",""0.0016699420154185848"",""0.0""]}"


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Calculamos las métricas principales
metricas = {}
for nombre_metrica in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=nombre_metrica
    )
    metricas[nombre_metrica] = evaluator.evaluate(predicciones)

print("Resultados del modelo: ")
print(f"  > Accuracy:            {metricas['accuracy']:.4f}")
print(f"  > F1 Score:            {metricas['f1']:.4f}")
print(f"  > Precision (media):   {metricas['weightedPrecision']:.4f}")
print(f"  > Recall (media):      {metricas['weightedRecall']:.4f}")


=== Resultados del modelo ===
  Accuracy:            0.9292
  F1 Score:            0.9108
  Precision (media):   0.9080
  Recall (media):      0.9292


---

## Paso 5: ¿Qué features son más importantes?

Random Forest nos dice cuánto contribuye cada variable a las predicciones. Esto es muy útil para **entender** el modelo, no solo usarlo.

In [0]:
import pandas as pd

# El modelo RF es la última etapa del pipeline
rf_model = modelo.stages[-1]

# Extraemos la importancia de cada feature
importancias = rf_model.featureImportances.toArray()

fi_pdf = (pd.DataFrame({"feature": todas_las_features, "importancia": importancias})
    .sort_values("importancia", ascending=False)
)

print("Top 10 features más importantes:")
for _, fila in fi_pdf.head(10).iterrows():
    barra = "█" * int(fila["importancia"] * 100)
    print(f"  {fila['feature']:25s} {fila['importancia']:.4f} {barra}")


Top 10 features más importantes:
  bwd_data_pkts_tot         0.2288 ██████████████████████
  service_idx               0.1435 ██████████████
  payload_bytes_per_second  0.1262 ████████████
  bwd_pkts_tot              0.0992 █████████
  flow_SYN_flag_count       0.0834 ████████
  fwd_data_pkts_tot         0.0827 ████████
  proto_idx                 0.0762 ███████
  flow_duration             0.0601 ██████
  fwd_header_size_tot       0.0395 ███
  flow_FIN_flag_count       0.0232 ██


---

# Parte III: Introducción a MLflow

## ¿Qué es MLflow?

**MLflow** es una plataforma open-source para gestionar el ciclo de vida de modelos de Machine Learning. Viene **integrada** en Databricks y resuelve un problema muy común: cuando entrenas muchos modelos con diferentes parámetros, ¿cómo llevas registro de qué probaste, qué resultados dio, y cuál fue el mejor?

MLflow tiene cuatro componentes principales:

| Componente | ¿Para qué sirve? |
|---|---|
| **Tracking** | Registrar parámetros, métricas y artefactos de cada experimento |
| **Models** | Empaquetar modelos en un formato estándar |
| **Model Registry** | Gestionar versiones de modelos (staging, producción…) |
| **Projects** | Empaquetar código ML para que sea reproducible |

En esta práctica nos centraremos en **Tracking**: registrar qué hicimos, con qué parámetros, y qué resultados obtuvimos.

---

## Paso 1: Configurar MLflow

En Databricks, MLflow ya está instalado. Solo necesitamos importarlo y configurar el experimento.

In [0]:
import mlflow
import mlflow.spark
import os

# Configuramos el nombre del experimento
# En Databricks, esto crea un experimento visible en la barra lateral (Experiments)
mlflow.set_experiment("/Users/{}/iot_attack_classification".format(
    spark.sql("SELECT current_user()").first()[0]
))

print(f"MLflow versión: {mlflow.__version__}")
print("Experimento configurado correctamente.")


TMP_UC = "/Volumes/workspace/default/test/mlflow_tmp"
os.environ["MLFLOW_DFS_TMP"] = TMP_UC

print("MLFLOW_DFS_TMP =", os.environ["MLFLOW_DFS_TMP"])

MLflow versión: 3.8.1
Experimento configurado correctamente.
MLFLOW_DFS_TMP = /Volumes/workspace/default/test/mlflow_tmp


---

## Paso 2: Entrenar y registrar un experimento

Ahora vamos a repetir el entrenamiento del modelo, pero esta vez **registrando todo en MLflow**: los parámetros del modelo, las métricas de evaluación, y el propio modelo entrenado.

Cada ejecución de `mlflow.start_run()` crea un **run** (ejecución) dentro del experimento.

In [0]:

if "modelo" not in globals():
    raise RuntimeError("No existe `modelo` en memoria. Ejecuta antes la celda 53.")

# Si no existen las predicciones, las generamos
if "predicciones" not in globals():
    predicciones = modelo.transform(test_df)

# Recalcular métricas por seguridad
metricas_v1 = {}
for nombre_metrica in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=nombre_metrica
    )
    metricas_v1[nombre_metrica] = evaluator.evaluate(predicciones)

NUM_TREES = 50
MAX_DEPTH = 8

# Arrancamos el modelo en MLFlow
with mlflow.start_run(run_name="random_forest_v1"):
    mlflow.log_param("num_trees", NUM_TREES)
    mlflow.log_param("max_depth", MAX_DEPTH)
    mlflow.log_param("num_features", len(todas_las_features))
    mlflow.log_param("features_categoricas", feature_cols_categorical)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_param("test_size", test_df.count())

    for nombre_metrica, valor in metricas_v1.items():
        mlflow.log_metric(nombre_metrica, valor)
        print(f"  {nombre_metrica}: {valor:.4f}")

    mlflow.spark.log_model(
        modelo,
        "modelo_rf",
        dfs_tmpdir="/Volumes/workspace/default/test"
    )

    print("\nRun registrado en MLflow correctamente.")

  accuracy: 0.9292
  f1: 0.9108
  weightedPrecision: 0.9080
  weightedRecall: 0.9292


2026/04/13 20:08:39 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/13 20:08:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-55e6a4eb-5c9b-443a-94cc-00/tmp5mk922vt/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/13 20:08:43 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l


Run registrado en MLflow correctamente.


---

## Paso 3: Probar con otros parámetros

La gracia de MLflow es poder **comparar** diferentes configuraciones. Vamos a entrenar un segundo modelo con más árboles y mayor profundidad para ver si mejora.

In [0]:
# Segundo experimento: más árboles y mayor profundidad
import gc

# Limpieza previa de referencias antiguas para no acumular modelos en la sesión
for var_name in [
    "rf_model",
    "predicciones",
    "modelo",
    "pipeline",
    "rf",
    "metricas_v1",
    "best_model",
    "loaded_model",
    "rf_v2",
    "pipeline_v2",
    "modelo_v2",
    "pred_v2",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

NUM_TREES_2 = 100
MAX_DEPTH_2 = 12

with mlflow.start_run(run_name="random_forest_v2"):

    mlflow.log_param("num_trees", NUM_TREES_2)
    mlflow.log_param("max_depth", MAX_DEPTH_2)
    mlflow.log_param("num_features", len(todas_las_features))
    mlflow.log_param("features_categoricas", feature_cols_categorical)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_param("test_size", test_df.count())

    rf_v2 = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=NUM_TREES_2,
        maxDepth=MAX_DEPTH_2,
        seed=42
    )

    pipeline_v2 = Pipeline(stages=indexers + [label_indexer, assembler, rf_v2])
    modelo_v2 = pipeline_v2.fit(train_df)

    pred_v2 = modelo_v2.transform(test_df)

    metricas_v2 = {}
    for nombre_metrica in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=nombre_metrica
        )
        valor = evaluator.evaluate(pred_v2)
        metricas_v2[nombre_metrica] = valor
        mlflow.log_metric(nombre_metrica, valor)
        print(f"  {nombre_metrica}: {valor:.4f}")

    mlflow.spark.log_model(
        spark_model=modelo_v2,
        artifact_path="modelo_rf",
        dfs_tmpdir=TMP_UC
    )

    print("\nRun registrado en MLflow correctamente.")

  accuracy: 0.9841
  f1: 0.9840
  weightedPrecision: 0.9842
  weightedRecall: 0.9841


2026/04/13 20:26:15 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/13 20:26:17 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-55e6a4eb-5c9b-443a-94cc-00/tmpavbffae1/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/13 20:26:17 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l


Run registrado en MLflow correctamente.


---

## Paso 4: Comparar experimentos

Podemos recuperar todos los runs de nuestro experimento y compararlos en una tabla.

> **Tip Databricks:** También puedes ir a la pestaña **Experiments** en la barra lateral y ver los resultados de forma visual, con gráficos de comparación.

In [0]:
# Recuperamos todos los runs del experimento
experimento = mlflow.get_experiment_by_name(
    "/Users/{}/iot_attack_classification".format(
        spark.sql("SELECT current_user()").first()[0]
    )
)

runs = mlflow.search_runs(experiment_ids=[experimento.experiment_id])

# Mostramos la comparación
columnas_interes = [
    "run_id", "params.num_trees", "params.max_depth",
    "metrics.accuracy", "metrics.f1", 
    "metrics.weightedPrecision", "metrics.weightedRecall"
]

# Filtramos solo columnas que existen
columnas_disponibles = [c for c in columnas_interes if c in runs.columns]
comparacion = runs[columnas_disponibles].sort_values("metrics.accuracy", ascending=False)

print("Comparación de modelos: ")
display(spark.createDataFrame(comparacion))


=== Comparación de modelos ===


run_id,params.num_trees,params.max_depth,metrics.accuracy,metrics.f1,metrics.weightedPrecision,metrics.weightedRecall
7884df628d9145c98a55c1be31696eb0,100,12,0.984057196860747,0.9840235347272974,0.9842447712520646,0.9840571968607471
773bddcd1b93458da03be9d47e50c6d5,100,12,0.984057196860747,0.9840235347272974,0.9842447712520646,0.9840571968607471
67f7dc6aae884a60ae3ae16ea10c7787,50,8,0.9292435386448618,0.9107905330581263,0.9079590243710386,0.9292435386448618


---

## Paso 5: Cargar y usar el mejor modelo

Una vez identificado el mejor run, podemos **cargar su modelo** directamente desde MLflow y usarlo para hacer predicciones nuevas. Esto es clave en producción: el modelo queda almacenado y versionado.

In [0]:
# Identificamos el mejor run (mayor accuracy)
mejor_run = comparacion.iloc[0]
mejor_run_id = mejor_run["run_id"]
print(f"Mejor run: {mejor_run_id}")
print(f"  Accuracy: {mejor_run.get('metrics.accuracy', 'N/A')}")

# Cargamos el modelo desde MLflow
modelo_cargado = mlflow.spark.load_model(f"runs:/{mejor_run_id}/modelo_rf")

# Lo usamos para predecir sobre datos nuevos (usamos test como ejemplo)
predicciones_nuevas = modelo_cargado.transform(test_df)

print(f"\nPredicciones generadas: {predicciones_nuevas.count()} filas")
display(
    predicciones_nuevas
    .select(target_col, "prediction", "probability")
    .limit(5)
)


Mejor run: 7884df628d9145c98a55c1be31696eb0
  Accuracy: 0.984057196860747



Predicciones generadas: 24337 filas


Attack_type,prediction,probability
MQTT_Publish,9.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.019720253045484838"",""0.021217992812784275"",""0.038719267605166856"",""0.3953575185603872"",""0.01049014937890803"",""0.017122104772158905"",""0.014681020733652312"",""0.01"",""0.0"",""0.4428481921774488"",""0.0"",""0.029843500914008753"",""0.0""]}"
MQTT_Publish,3.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.019720253045484835"",""0.019840630436971457"",""0.03128942721344388"",""0.38466674460783773"",""0.014551814175840543"",""0.17027349167991937"",""0.0"",""0.0"",""2.4207213749697407E-6"",""0.005038446513557944"",""1.2103606874848703E-5"",""0.35460466799869444"",""0.0""]}"
MQTT_Publish,3.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.019720253045484835"",""0.019840630436971457"",""0.03128942721344388"",""0.38466674460783773"",""0.014551814175840543"",""0.17027349167991937"",""0.0"",""0.0"",""2.4207213749697407E-6"",""0.005038446513557944"",""1.2103606874848703E-5"",""0.35460466799869444"",""0.0""]}"
Thing_Speak,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.5822749951598072"",""0.0936004391047924"",""5.811114569284503E-4"",""0.3229985024816356"",""3.631082062454611E-5"",""0.0"",""0.0"",""8.958579459103081E-5"",""2.9283204795351093E-4"",""8.507087029277846E-5"",""4.115226337448559E-5"",""0.0""]}"
Thing_Speak,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.5822749951598072"",""0.0936004391047924"",""5.811114569284503E-4"",""0.3229985024816356"",""3.631082062454611E-5"",""0.0"",""0.0"",""8.958579459103081E-5"",""2.9283204795351093E-4"",""8.507087029277846E-5"",""4.115226337448559E-5"",""0.0""]}"


---

## Conclusiones

En esta práctica hemos cubierto tres pilares del análisis de datos distribuido:

**DataFrames** nos han permitido cargar, explorar, filtrar, agrupar y transformar datos de tráfico IoT de forma concisa y eficiente. Las operaciones como `groupBy`, `agg`, `join` y `withColumn` son las herramientas fundamentales del día a día con Spark.

**MLlib** nos ha permitido construir un pipeline completo de clasificación: desde la preparación de features hasta la evaluación del modelo, pasando por la transformación de datos categóricos y el entrenamiento de un Random Forest.

**MLflow** nos ha enseñado a gestionar experimentos de ML de forma profesional: registrar parámetros y métricas, comparar configuraciones, y guardar/cargar modelos. Esta es una práctica esencial en cualquier proyecto real de Machine Learning.

---

***
[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/)

This work has been created by Pablo C. Cañizares. This work is licensed under a [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/).